# Original Gemini 2.5 Flash run (main analysis, 14,838 reviews)
Input: `marakech.csv` (collected reviews; not redistributed). Output: the `llm_*` columns released in `data/corpus_llm_outputs.csv`.
Settings: batches of 15 reviews, 4 parallel requests, 3 retries, default generation settings (no temperature set), each review trimmed and truncated to 500 characters. The prompt is in `prompt.txt` (Figure 1 of the paper).

In [ ]:
   !pip install google-generativeai

In [ ]:
import pandas as pd
# Load the collected reviews (not redistributed: TripAdvisor terms of service)
df = pd.read_csv("marakech.csv")
df.shape


In [ ]:
import time
import json
import pandas as pd
from tqdm import tqdm
import os
import google.generativeai as genai
from concurrent.futures import ThreadPoolExecutor, as_completed

# ========== CONFIG ==========
BATCH_SIZE  = 15      # 15 reviews per request (plus de champs = moins par batch)
MAX_WORKERS = 4       # 4 parallel requests
SAVE_EVERY  = 10

from getpass import getpass
genai.configure(api_key=getpass("Gemini API key: "))  # typed at run time, never stored
model = genai.GenerativeModel('gemini-2.5-flash')

# ========== LOAD DATA ==========
if os.path.exists("progress_backup_v2.csv"):
    df = pd.read_csv("progress_backup_v2.csv")
    print("Resuming from backup: " + str(len(df)) + " rows")
else:
    df = df.dropna(subset=['text'])
    df = df.reset_index(drop=True)
    print("Starting fresh: " + str(len(df)) + " rows")

# ========== COLONNES DE SORTIE ==========
new_cols = [
    "llm_sentiment",       # Positive / Negative / Neutral
    "llm_score",           # 1-5
    "llm_emotion",         # Joy / Anger / Fear / ...
    "llm_safety",          # Positive / Negative / Neutral / Not_mentioned
    "llm_pricing",         # ...
    "llm_service",         # ...
    "llm_cleanliness",     # ...
    "llm_atmosphere",      # ...
    "llm_problems",        # JSON list
    "llm_price_perception",# Expensive / Fair / Cheap / Not_mentioned
    "llm_recommendation",  # Yes / No / Not_mentioned
    "llm_revisit",         # Yes / No / Not_mentioned
    "llm_keywords",        # JSON list
]

for col in new_cols:
    if col not in df.columns:
        df[col] = None

to_process = [
    i for i in range(len(df))
    if pd.isna(df.loc[i, "llm_sentiment"])
]
print("Rows to process: " + str(len(to_process)))

# ========== CLASSIFICATION FUNCTION ==========
def classify_batch(reviews):
    cleaned = []
    for r in reviews:
        if not isinstance(r, str):
            r = ""
        r = r.strip()[:500]
        cleaned.append(r)

    reviews_json = json.dumps(cleaned, ensure_ascii=False)

    prompt = """You are a tourism analyst. Analyze each TripAdvisor review and return ONLY a JSON array (same length and order as input). No explanation, no markdown.

For each review, return:
{
  "overall_sentiment": "Positive" | "Negative" | "Neutral",
  "sentiment_score": 1-5,
  "emotional_tone": "Joy" | "Anger" | "Fear" | "Disgust" | "Sadness" | "Surprise" | "Trust" | "Frustration" | "Neutral" | "Other",
  "aspects": {
    "safety": "Positive" | "Negative" | "Neutral" | "Not_mentioned",
    "pricing": "Positive" | "Negative" | "Neutral" | "Not_mentioned",
    "service": "Positive" | "Negative" | "Neutral" | "Not_mentioned",
    "cleanliness": "Positive" | "Negative" | "Neutral" | "Not_mentioned",
    "atmosphere": "Positive" | "Negative" | "Neutral" | "Not_mentioned"
  },
  "problems": ["Aggressive_vendors" | "Navigation_difficulty" | "Getting_lost" | "Bad_smell" | "Overcrowding" | "Poor_condition" | "High_prices" | "Poor_cleanliness" | "Scam" | "Safety" | "Other"],
  "economic_signals": {
    "price_perception": "Expensive" | "Fair" | "Cheap" | "Not_mentioned",
    "recommendation": "Yes" | "No" | "Not_mentioned",
    "revisit_intention": "Yes" | "No" | "Not_mentioned"
  },
  "keywords": ["keyword1", "keyword2", "keyword3"]
}

Rules:
- If no problems, return "problems": []
- Maximum 3 problems per review
- Maximum 3-5 keywords per review
- Translate any non-English problems/keywords into English
- Be consistent: same issue = same wording across all reviews
- Return ONLY the JSON array, nothing else

Examples:
"Beautiful place but shopkeepers are aggressive and everything is overpriced"
→ {"overall_sentiment": "Negative", "sentiment_score": 2, "emotional_tone": "Frustration", "aspects": {"safety": "Not_mentioned", "pricing": "Negative", "service": "Negative", "cleanliness": "Not_mentioned", "atmosphere": "Positive"}, "problems": ["Aggressive_vendors", "High_prices"], "economic_signals": {"price_perception": "Expensive", "recommendation": "Not_mentioned", "revisit_intention": "Not_mentioned"}, "keywords": ["beautiful", "aggressive", "overpriced"]}

"Amazing experience, felt very safe, would definitely come back!"
→ {"overall_sentiment": "Positive", "sentiment_score": 5, "emotional_tone": "Joy", "aspects": {"safety": "Positive", "pricing": "Not_mentioned", "service": "Not_mentioned", "cleanliness": "Not_mentioned", "atmosphere": "Positive"}, "problems": [], "economic_signals": {"price_perception": "Not_mentioned", "recommendation": "Yes", "revisit_intention": "Yes"}, "keywords": ["amazing", "safe", "come back"]}

Reviews:
""" + reviews_json + """

Return ONLY the JSON array."""

    for attempt in range(3):
        try:
            response = model.generate_content(prompt)
            content = response.text.strip()
            content = content.replace("```json", "").replace("```", "").strip()
            data = json.loads(content)
            if not isinstance(data, list) or len(data) != len(cleaned):
                raise ValueError("Length mismatch: expected " + str(len(cleaned)) + " got " + str(len(data)))
            return data
        except Exception as e:
            print("Retry " + str(attempt + 1) + ": " + str(e)[:80])
            time.sleep(3)

    return [{"overall_sentiment": None, "sentiment_score": None, "emotional_tone": None,
             "aspects": {"safety": None, "pricing": None, "service": None, "cleanliness": None, "atmosphere": None},
             "problems": [], "economic_signals": {"price_perception": None, "recommendation": None, "revisit_intention": None},
             "keywords": []} for _ in cleaned]

def process_chunk(chunk_data):
    indices, reviews = chunk_data
    results = classify_batch(reviews)
    return indices, results

# ========== EXTRACT FIELDS ==========
def extract_fields(res):
    """Extraire les champs du résultat JSON en colonnes plates"""
    fields = {}
    fields["llm_sentiment"]        = res.get("overall_sentiment")
    fields["llm_score"]            = res.get("sentiment_score")
    fields["llm_emotion"]          = res.get("emotional_tone")

    aspects = res.get("aspects", {})
    fields["llm_safety"]           = aspects.get("safety")
    fields["llm_pricing"]          = aspects.get("pricing")
    fields["llm_service"]          = aspects.get("service")
    fields["llm_cleanliness"]      = aspects.get("cleanliness")
    fields["llm_atmosphere"]       = aspects.get("atmosphere")

    fields["llm_problems"]         = json.dumps(res.get("problems", []))

    eco = res.get("economic_signals", {})
    fields["llm_price_perception"] = eco.get("price_perception")
    fields["llm_recommendation"]   = eco.get("recommendation")
    fields["llm_revisit"]          = eco.get("revisit_intention")

    fields["llm_keywords"]         = json.dumps(res.get("keywords", []))

    return fields

# ========== MAIN LOOP ==========
if len(to_process) > 0:

    batches = []
    for b in range(0, len(to_process), BATCH_SIZE):
        idx = to_process[b:b + BATCH_SIZE]
        rev = [df.loc[i, "text"] for i in idx]
        batches.append((idx, rev))

    print("Total batches : " + str(len(batches)))
    print("Batch size    : " + str(BATCH_SIZE))
    print("Workers       : " + str(MAX_WORKERS))
    print("Estimated time: ~" + str(round(len(batches) / MAX_WORKERS * 10 / 3600, 1)) + " hours")

    batch_count = 0

    with tqdm(total=len(batches), desc="LLM Analysis") as pbar:
        for chunk_start in range(0, len(batches), MAX_WORKERS):
            chunk = batches[chunk_start:chunk_start + MAX_WORKERS]

            with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
                futures = {executor.submit(process_chunk, c): c for c in chunk}

                for future in as_completed(futures):
                    try:
                        indices, results = future.result()
                        for idx, res in zip(indices, results):
                            fields = extract_fields(res)
                            for col, val in fields.items():
                                df.loc[idx, col] = val
                    except Exception as e:
                        print("Error: " + str(e)[:80])

                    batch_count += 1
                    pbar.update(1)

            # Sauvegarde périodique
            if batch_count % SAVE_EVERY == 0:
                df.to_csv("progress_backup_v2.csv", index=False)
                done = min(batch_count * BATCH_SIZE, len(to_process))
                print("Saved — " + str(done) + " / " + str(len(to_process)))

            time.sleep(1)

    # Sauvegarde finale
    df.to_csv("LLM_FULL_ANALYSIS.csv", index=False, encoding="utf-8-sig")
    print("\n" + "=" * 60)
    print("Done! Saved as LLM_FULL_ANALYSIS.csv")
    print("=" * 60)

    # Statistiques rapides
    print("\n📊 Statistiques rapides :")
    print("\n--- Sentiment ---")
    print(df['llm_sentiment'].value_counts())
    print("\n--- Emotional Tone ---")
    print(df['llm_emotion'].value_counts())
    print("\n--- Aspects (Safety) ---")
    print(df['llm_safety'].value_counts())
    print("\n--- Aspects (Pricing) ---")
    print(df['llm_pricing'].value_counts())
    print("\n--- Aspects (Service) ---")
    print(df['llm_service'].value_counts())
    print("\n--- Aspects (Cleanliness) ---")
    print(df['llm_cleanliness'].value_counts())
    print("\n--- Aspects (Atmosphere) ---")
    print(df['llm_atmosphere'].value_counts())
    print("\n--- Price Perception ---")
    print(df['llm_price_perception'].value_counts())
    print("\n--- Recommendation ---")
    print(df['llm_recommendation'].value_counts())
    print("\n--- Revisit Intention ---")
    print(df['llm_revisit'].value_counts())

    # Top Problems
    print("\n--- Top Problems ---")
    all_problems = []
    for p in df['llm_problems'].dropna():
        try:
            problems = json.loads(p)
            all_problems.extend(problems)
        except:
            pass
    problem_counts = pd.Series(all_problems).value_counts()
    print(problem_counts.head(15))

    # Top Keywords
    print("\n--- Top Keywords ---")
    all_keywords = []
    for k in df['llm_keywords'].dropna():
        try:
            keywords = json.loads(k)
            all_keywords.extend(keywords)
        except:
            pass
    keyword_counts = pd.Series(all_keywords).value_counts()
    print(keyword_counts.head(20))

    # Sample
    print("\n📝 Sample results:")
    sample_cols = ["text", "llm_sentiment", "llm_emotion", "llm_problems", "llm_price_perception"]
    print(df[sample_cols].head(3).to_string())

else:
    print("Nothing to process — all rows already analyzed!")